# Clustering

## Summary of findings

Present a summary of your findings here, including the answers to the questions from the exercise sheet. Then present your code and additional findings below.

You may use sklearn’s `KMeans` for Lloyd’s algorithm, but you must implement k-Means++ and the coreset sampling procedure yourself.

## Clustering using Lloyd's algorithm

In [4]:
from PIL.ImageChops import difference
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.datasets import mnist
import numpy as np

# Load MNIST or Fashion-MNIST
(x_train, _), _ = mnist.load_data()
# (x_train, _), _ = fashion_mnist.load_data() # use this line to load Fashion MNIST

# Flatten and scale images
X = x_train.reshape(len(x_train), -1)  # shape: (60000, 784)
X = StandardScaler().fit_transform(X)
print(X)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [5]:
# K-Means clustering
kmeans = KMeans(n_clusters=10)
kmeans.fit(X)

# Output objective function value (inertia_)
print(f'K-Means objective (inertia): {kmeans.inertia_:.2f}')

K-Means objective (inertia): 36371138.69


## Implementation of $k$-Means++

In [10]:
def k_means_plus_plus(X, k):
  x = X[np.random.randint(0, X.shape[0])]
  T = [x]
  min_distances = np.full(X.shape[0], np.inf)
  while len(T) < k: # happens k times
    # calculate the distances to the newest center
    z = T[-1]
    new_distances = np.sum((X - z) ** 2, axis=1) # O(n*d)

    # find the cost(x, T) for all X
    # cost_x = np.min(cost_for_each_point, axis=0) # O(n*k)
    cost_x = np.minimum(min_distances, new_distances) # O(n)
    cost_T = np.sum(cost_x) # O(n)

    # randomly select a new point with the calculated probability
    sampled_index = np.random.choice(X.shape[0], p=(cost_x/cost_T)) # O(n)
    T.append(X[sampled_index])
  return T

coresets = [k_means_plus_plus(X, 3),
            k_means_plus_plus(X,6),
            k_means_plus_plus(X,10)]

## Experimental Evaluation